<a href="https://colab.research.google.com/github/adimika04-collab/Proses-Stokastik/blob/main/Project_UAS_Prostok.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import random

random.seed(123)
np.random.seed(123)

class MarkovChain:

    def __init__(self, states, transition_matrix):

        self.states = states
        self.transition_matrix = np.array(transition_matrix)
        self.num_states = len(states)

        if self.transition_matrix.shape != (self.num_states, self.num_states):
            raise ValueError("Ukuran matriks tidak sesuai.")

        if not np.allclose(self.transition_matrix.sum(axis=1), 1):
            raise ValueError("Jumlah probabilitas setiap baris harus 1.")

        self.state_to_index = {
            state: i for i, state in enumerate(states)
        }

        self.index_to_state = {
            i: state for i, state in enumerate(states)
        }

    # ==================================================
    # MENENTUKAN STATE BERIKUTNYA
    # ==================================================

    def get_next_state(self, current_state):

        idx = self.state_to_index[current_state]

        next_idx = random.choices(
            range(self.num_states),
            weights=self.transition_matrix[idx],
            k=1
        )[0]

        return self.index_to_state[next_idx]

    # ==================================================
    # SIMULASI SATU PASIEN
    # ==================================================

    def simulate(self, initial_state, num_steps):

        current_state = initial_state

        history = [current_state]

        for step in range(num_steps):

            current_state = self.get_next_state(current_state)

            history.append(current_state)

            if current_state == "Meninggal":
                break

        return history

    # ==================================================
    # PROBABILITAS TIAP HARI
    # ==================================================

    def probability_table(self, initial_state, num_steps):

        initial_vector = np.zeros(self.num_states)

        initial_vector[
            self.state_to_index[initial_state]
        ] = 1

        current_prob = initial_vector

        results = []

        for day in range(num_steps + 1):

            row = {"Hari": day}

            for state, prob in zip(
                self.states,
                current_prob
            ):
                row[state] = round(prob, 4)

            results.append(row)

            current_prob = (
                current_prob @ self.transition_matrix
            )

        return pd.DataFrame(results)

    # ==================================================
    # DISTRIBUSI JANGKA PANJANG
    # ==================================================

    def steady_state(self, initial_state, n=100):

        initial_vector = np.zeros(self.num_states)

        initial_vector[
            self.state_to_index[initial_state]
        ] = 1

        Pn = np.linalg.matrix_power(
            self.transition_matrix,
            n
        )

        steady = initial_vector @ Pn

        return dict(zip(self.states, steady))

    # ==================================================
    # SIMULASI BANYAK PASIEN
    # ==================================================

    def simulate_many_patients(
        self,
        initial_state,
        num_steps,
        num_patients=100
    ):

        final_states = []

        for _ in range(num_patients):

            history = self.simulate(
                initial_state,
                num_steps
            )

            final_states.append(
                history[-1]
            )

        result = (
            pd.Series(final_states)
            .value_counts(normalize=True)
            .sort_index()
        )

        return result


# ==================================================
# STATE KONDISI PASIEN
# ==================================================

states = [
    "Sehat",
    "Sakit Ringan",
    "Sakit Berat",
    "Meninggal"
]

# ==================================================
# MATRIKS TRANSISI
# ==================================================

transition_matrix = [

    # Sehat
    [0.85, 0.10, 0.04, 0.01],

    # Sakit Ringan
    [0.50, 0.30, 0.15, 0.05],

    # Sakit Berat
    [0.10, 0.25, 0.40, 0.25],

    # Meninggal (absorbing state)
    [0.00, 0.00, 0.00, 1.00]
]

# ==================================================
# MEMBUAT MODEL
# ==================================================

mc = MarkovChain(
    states,
    transition_matrix
)

# ==================================================
# SIMULASI 1 PASIEN
# ==================================================

print("\n=== SIMULASI SATU PASIEN ===")

history = mc.simulate(
    initial_state="Sehat",
    num_steps=10
)

for day, state in enumerate(history):
    print(f"Hari ke-{day}: {state}")

# ==================================================
# TABEL PROBABILITAS
# ==================================================

print("\n=== TABEL PROBABILITAS ===")

df_prob = mc.probability_table(
    initial_state="Sehat",
    num_steps=10
)

print(df_prob)

# ==================================================
# DISTRIBUSI JANGKA PANJANG
# ==================================================

print("\n=== DISTRIBUSI JANGKA PANJANG ===")

steady = mc.steady_state(
    initial_state="Sehat",
    n=100
)

for state, prob in steady.items():
    print(f"{state:<15}: {prob:.4f}")




=== SIMULASI SATU PASIEN ===
Hari ke-0: Sehat
Hari ke-1: Sehat
Hari ke-2: Sehat
Hari ke-3: Sehat
Hari ke-4: Sehat
Hari ke-5: Sakit Ringan
Hari ke-6: Sehat
Hari ke-7: Sehat
Hari ke-8: Sehat
Hari ke-9: Sakit Ringan
Hari ke-10: Sehat

=== TABEL PROBABILITAS ===
    Hari   Sehat  Sakit Ringan  Sakit Berat  Meninggal
0      0  1.0000        0.0000       0.0000     0.0000
1      1  0.8500        0.1000       0.0400     0.0100
2      2  0.7765        0.1250       0.0650     0.0335
3      3  0.7290        0.1314       0.0758     0.0638
4      4  0.6930        0.1313       0.0792     0.0966
5      5  0.6626        0.1285       0.0791     0.1299
6      6  0.6353        0.1246       0.0774     0.1627
7      7  0.6101        0.1203       0.0751     0.1946
8      8  0.5862        0.1158       0.0725     0.2255
9      9  0.5634        0.1115       0.0698     0.2553
10    10  0.5416        0.1072       0.0672     0.2839

=== DISTRIBUSI JANGKA PANJANG ===
Sehat          : 0.0158
Sakit Ringan   : 0.00